# Itemset on Google Colab

Run all cells on a CPU runtime. The notebook resolves the public repository to an exact commit, verifies Julia 1.13.2, Java 25 and the pinned SPMF JAR, then runs the Julia demo. Setup and application data downloads can take several minutes.

The demo checks the package, mines baskets and Semeion images, and compares both variants with an independent SPMF run on Chess. The Chess check measures peak process RSS for one threshold. The full benchmark series have their own commands and figures in the [report guide](https://github.com/NgTHung/Itemset/tree/main/docs/report).

In [ ]:
import os
from pathlib import Path
import runpy
import subprocess
import tempfile
from urllib.parse import quote
import urllib.request

ITEMSET_REVISION = ""
repository = os.environ.get("ITEMSET_REPOSITORY", "https://github.com/NgTHung/Itemset")
github_prefix = "https://github.com/"
if not repository.startswith(github_prefix):
    raise ValueError("ITEMSET_REPOSITORY must be a GitHub HTTPS repository URL")
repository_slug = repository.removeprefix(github_prefix).rstrip("/").removesuffix(".git")
revision = ITEMSET_REVISION or os.environ.get("ITEMSET_REVISION")
if not revision:
    revision = subprocess.check_output(
        ["git", "ls-remote", "--exit-code", repository, "refs/heads/main"],
        text=True, timeout=60,
    ).split()[0]

workspace = Path(tempfile.mkdtemp(prefix="itemset-colab-", dir="/content"))
runner_path = workspace / "run_colab_notebook.py"
runner_url = f"https://raw.githubusercontent.com/{repository_slug}/{quote(revision, safe='')}/scripts/run_colab_notebook.py"
with urllib.request.urlopen(runner_url, timeout=60) as response:
    runner_path.write_bytes(response.read())
evidence = workspace / "evidence"
print(f"Revision: {revision}", flush=True)
runner = runpy.run_path(str(runner_path))
if runner["main"]([
    "--repository", repository,
    "--revision", revision,
    "--root", str(workspace / "repo"),
    "--evidence", str(evidence),
]):
    raise RuntimeError("Demo failed. See the evidence archive in the cell output.")

## Semeion patterns

The examples below compare source images that contain a visual-word pattern with images that lack one or more of its words. Use the Colab Files sidebar to download the evidence ZIP. It includes the full itemset maps, source image IDs, basket rules, SPMF comparisons and peak RSS measurements.

In [ ]:
from IPython.display import SVG, display

semeion = evidence / "semeion"
display(SVG(filename=str(semeion / "visual_words.svg")))
display(SVG(filename=str(semeion / "supporting-0.05.svg")))